# ML-08 — Capstone Modeling: Lane 2 (Refresh / Content Opportunity Scoring)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu-yadav-10/Flyrank-ML-starter-template/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This week I go from a hand-written rule (Week 4) to a learned model, and I hold the model to the same
**data, split, and metric** I used for the baseline: precision@50 on a **client-held-out** test split.

**Structure (filled in order):**
1. Method choice and why
2. Split design
3. Train + compare vs my Week-4 baseline (same split, same metric)
4. Errors and interpretation (feature sanity, error buckets, 3 wrong cases)
5. Self-check

In [1]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path

# ---------- repo root (robust to launch directory) ----------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("Flyrank-ML-starter-template"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/himanshu-yadav-10/Flyrank-ML-starter-template",
                        "Flyrank-ML-starter-template"], check=True)
    os.chdir("Flyrank-ML-starter-template")
    REPO = Path(os.getcwd())
else:
    def _find_repo_root(start: str):
        here = Path(start).resolve()
        for _ in range(10):
            if (here / "data" / "raw" / "content_refresh_anonymized.csv").exists():
                return here
            if here.parent == here:
                break
            here = here.parent
        for sub in Path(start).resolve().iterdir():
            if sub.is_dir() and (sub / "data" / "raw" / "content_refresh_anonymized.csv").exists():
                return sub
        return None

    REPO = _find_repo_root(os.getcwd())
    assert REPO is not None, "Starter CSV not found - run this notebook from the repo tree."

sys.path.insert(0, str(REPO / "scripts"))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k

DATA = REPO / "data" / "processed" / "refresh_feature_vector.csv"
OUT = REPO / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

df = pd.read_csv(DATA)
print("Loaded:", len(df), "rows x", len(df.columns), "cols")
print("Target rate (is_declining_label):", round(df["is_declining_label"].mean(), 4))
print("Clients:", df["client_id"].nunique())
print("Feature lists from scripts/ml_utils.py (authoritative):")
print("  numeric   :", len(MODEL_NUMERIC_FEATURES), "features")
print("  categorical:", len(MODEL_CATEGORICAL_FEATURES), "features")
print("scikit-learn", __import__('sklearn').__version__, "| seed", RANDOM_STATE)

# ---- leakage guard: the feature lists must never touch label-derived columns ----
LEAK_COLS = ["trend_direction", "trend_pct", "is_declining_label"]
features = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
assert not any(c in features for c in LEAK_COLS), f"Leak in feature lists: {set(features) & set(LEAK_COLS)}"
print("Leak guard: no trend_direction / trend_pct / label in the feature lists.")

Loaded: 30000 rows x 52 cols
Target rate (is_declining_label): 0.5421
Clients: 32
Feature lists from scripts/ml_utils.py (authoritative):
  numeric   : 18 features
  categorical: 8 features


scikit-learn 1.6.1 | seed 42
Leak guard: no trend_direction / trend_pct / label in the feature lists.


---
## 1. Method choice and why

**The question is 'which pages first?'** so the honest evaluation shape is a *ranked score* measured at the top of the queue.
Following the week's skill table (yes/no + observed label \-> start with Logistic Regression, then Random Forest,
evaluated as a ranking via precision@K), I train **four** classifiers:

- **Logistic Regression** \- the readable linear anchor. Weak single signals mean linear trade-offs probably plateau quickly, but it is the cheapest honest baseline to beat.
- **Decision Tree** (depth 5) \- printable and interpretable, so I can sanity-check what the split surfaces (position and content-type interactions).
- **Random Forest** \- primary candidate: captures the interactions Week 2 measured (decline peaks in *striking/page_1*, collapses in *top_3*; feedly\-top_3 almost never declines).
- **Gradient Boosting** (sklearn, safe) \- included only to test whether extra complexity earns a seat; the skill says a model 2 points stronger at the cost of opacity must *earn* it.

**Target:** `is_declining_label` (54.2% of rows). **Score per page:** `P(declining | signals)`.
**Primary metric: precision@50 on held-out clients** (matches the action: the team works the top of the queue), with precision@20/@100, ROC-AUC, average precision, and F1 reported alongside.

In [2]:
# --- method confirmation: distributions we are trying to separate ---
base_rate = df["is_declining_label"].mean()
print("Base rate (random ordering upper bound):", round(base_rate, 4))
print("Declining rows:", int(df["is_declining_label"].sum()), "/", len(df))
print()
print("Per-client pages (the imbalance that forces a grouped split):")
print(df.groupby("client_id")["content_id"].nunique().describe().round(1).to_string())
print()
print("Weak single signals - no single feature separates decline well:")
from sklearn.metrics import roc_auc_score
for c in ["content_age_days", "ctr", "engagement_rate", "avg_position"]:
    s = pd.to_numeric(df[c], errors="coerce").fillna(0)
    print(f"  {c:<16} AUC={roc_auc_score(df['is_declining_label'], s):.3f}")

Base rate (random ordering upper bound): 0.5421
Declining rows: 16262 / 30000

Per-client pages (the imbalance that forces a grouped split):
count      32.0
mean      937.5
std      1376.4
min         3.0
25%       110.2
50%       567.0
75%      1058.8
max      7008.0

Weak single signals - no single feature separates decline well:
  content_age_days AUC=0.409
  ctr              AUC=0.515
  engagement_rate  AUC=0.496
  avg_position     AUC=0.530


---
## 2. Split design

**Grouped by client (client-holdout), 80/20, seed 42.**

Rows from the same client are not independent: they share a keyword space, an update cadence, and a site context.
A row-level random split would quietly leak each client's habits into the training rows of that same client
(and would look artificially great). Holding out **whole clients** tests the real claim: *does this model order
pages correctly for a client it has never seen?* That is the deployment condition.

All metrics below are computed **once**, on this fixed test fold, by every method including the Week-4 rule.
The Week-4 rule's visibility *percentile* is re-fit on the **training split** only (via interpolation), so the
baseline never touches the test distribution for its normalization.

In [3]:
# --- grouped split by client ---
rng = np.random.default_rng(RANDOM_STATE)
clients = df["client_id"].drop_duplicates().to_numpy()
shuffled = rng.permutation(clients)
n_test = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test])
test_mask = df["client_id"].isin(test_clients).to_numpy()
train_mask = ~test_mask

for name, m in [("train", train_mask), ("test", test_mask)]:
    y = df.loc[m, "is_declining_label"]
    print(f"{name:<6} rows={int(m.sum()):>6}  clients={df.loc[m,'client_id'].nunique():>2}  "
          f"declining_rate={y.mean():.3f}  classes={sorted(y.unique())}")

assert train_mask.sum() > 0 and test_mask.sum() > 0
assert df.loc[train_mask, "is_declining_label"].nunique() == 2
assert df.loc[test_mask, "is_declining_label"].nunique() == 2
print("Split OK: both classes present in train and test.")

train  rows= 27675  clients=26  declining_rate=0.555  classes=[np.int64(0), np.int64(1)]
test   rows=  2325  clients= 6  declining_rate=0.391  classes=[np.int64(0), np.int64(1)]
Split OK: both classes present in train and test.


---
## 3. Train + compare vs my Week-4 baseline

Same data file, same client-held-out test fold, same top-queue metric (precision@K).
The two things compared:
- **Week-4 rule (mine)** \- `stale_flag x visible_flag x visibility_pctrank`, the exact rule committed in
  `work/notebooks/w04_baseline_score.ipynb` (stale \>= 180 days since update, visible \>= 500 impressions).
- **Four trained classifiers** on the leakage-safe feature list from `scripts/ml_utils.py`.

Winner = best precision@50 on the held-out fold. A model that wins only at a different K, or only by a hair,
reports both numbers as the finding.

In [4]:
# --- feature matrix (identical build for every model) ---
num = [c for c in MODEL_NUMERIC_FEATURES if c in df.columns]
Xnum = df[num].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat = [c for c in MODEL_CATEGORICAL_FEATURES if c in df.columns]
Xcat = pd.get_dummies(df[cat].fillna("unknown").astype(str), prefix=cat, dtype=float)
X = pd.concat([Xnum.reset_index(drop=True), Xcat.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)
assert not any(c in X.columns for c in LEAK_COLS), "leakage column in feature matrix"
assert len(X) == len(df) == len(y)
print("Feature matrix:", X.shape)
print("No label-derived or future-window columns in features (verified above).")

Xtr, Xte = X[train_mask].reset_index(drop=True), X[test_mask].reset_index(drop=True)
ytr, yte = y[train_mask].reset_index(drop=True), y[test_mask].reset_index(drop=True)

Feature matrix: (30000, 52)
No label-derived or future-window columns in features (verified above).


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score,
)

MODELS = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
        n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
    "gradient_boosting": GradientBoostingClassifier(
        max_depth=3, n_estimators=150, learning_rate=0.1, random_state=RANDOM_STATE),
}

def proba(model, X):
    return np.asarray(model.predict_proba(X))[:, 1]

def quick_metrics(y, scores):
    preds = (scores >= 0.5).astype(int)
    m = {
        "precision_at_20": precision_at_k(y, scores, 20),
        "precision_at_50": precision_at_k(y, scores, 50),
        "precision_at_100": precision_at_k(y, scores, 100),
        "roc_auc": roc_auc_score(y, scores),
        "average_precision": average_precision_score(y, scores),
        "f1": f1_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0),
        "accuracy": accuracy_score(y, preds),
    }
    return m

test_scores = {}
model_results = {}
for name, model in MODELS.items():
    model.fit(Xtr, ytr)
    s = proba(model, Xte)
    test_scores[name] = s
    model_results[name] = quick_metrics(yte, s)
    print(f"fitted {name:<20}  p@50={model_results[name]['precision_at_50']:.3f}")

fitted logistic_regression   p@50=0.400
fitted decision_tree         p@50=0.640


fitted random_forest         p@50=0.740


fitted gradient_boosting     p@50=0.860


In [6]:
# --- my Week-4 rule, re-fit on train only, scored on the same test fold ---
train_logsorted = np.sort(np.log1p(
    pd.to_numeric(df.loc[train_mask, "impressions_90d"], errors="coerce").fillna(0).to_numpy()))
u = np.linspace(0., 1., len(train_logsorted))

def w4_baseline_scores(frame):
    log_imp = np.log1p(pd.to_numeric(frame["impressions_90d"], errors="coerce").fillna(0).to_numpy())
    vis = np.interp(log_imp, train_logsorted, u)  # visibility percentile, fit on TRAIN only
    stale = (pd.to_numeric(frame["days_since_last_update"], errors="coerce").fillna(0) >= 180).astype(float)
    visible = (pd.to_numeric(frame["impressions_90d"], errors="coerce").fillna(0) >= 500).astype(float)
    return stale * visible * vis

w4_test = w4_baseline_scores(df.iloc[test_mask])
w4_metrics = quick_metrics(yte, w4_test)
print("Week-4 rule on the test fold:")
print(f"  flagged (score > 0)      = {(w4_test > 0).sum()} of {len(w4_test)}")
print(f"  precision@50             = {w4_metrics['precision_at_50']:.3f}")

# --------------------------- comparison table ---------------------------
rows = ["model_or_rule"] + [
    "precision_at_20", "precision_at_50", "precision_at_100",
    "roc_auc", "average_precision", "f1",
]
results = {}
base_rate_fold = float(yte.mean())
results["base_rate"] = {
    "precision_at_20": base_rate_fold, "precision_at_50": base_rate_fold,
    "precision_at_100": base_rate_fold, "roc_auc": 0.5,
    "average_precision": base_rate_fold, "f1": base_rate_fold}
results["w4_rule_baseline"] = {k: float(w4_metrics[k]) for k in [
    "precision_at_20", "precision_at_50", "precision_at_100", "roc_auc", "average_precision", "f1"]}
for name, m in model_results.items():
    results[name] = {k: float(m[k]) for k in ["precision_at_20", "precision_at_50", "precision_at_100", "roc_auc", "average_precision", "f1"]}

table_rows = ["method     | p@20  | p@50  | p@100 | roc_auc | avg_prec | f1"]
for name, m in [("base_rate   ", results["base_rate"]), (
    "w4_rule     ", results["w4_rule_baseline"]
)] + [(n[:11]+" "*(11-len(n[:11])), m) for n, m in results.items() if n not in ("base_rate", "w4_rule_baseline")]:
    table_rows.append(f"{name:12} | {m['precision_at_20']:.3f} | {m['precision_at_50']:.3f} | "
                      f"{m['precision_at_100']:.3f} | {m['roc_auc']:.3f} | {m['average_precision']:.3f} | {m['f1']:.3f}")

print()
for line in table_rows:
    print(line)

# winner by precision@50
winner = max(model_results, key=lambda n: model_results[n]["precision_at_50"])
print()
print("Winner by precision@50:", winner, "=", round(model_results[winner]["precision_at_50"], 3))
print("Week-4 rule precision@50:", round(w4_metrics["precision_at_50"], 3))
print("Base rate (test fold):", round(base_rate_fold, 3))
print(f"Lift over base rate: {model_results[winner]['precision_at_50'] / base_rate_fold:.2f}x")
print(f"Lift over Week-4 rule: {model_results[winner]['precision_at_50'] / w4_metrics['precision_at_50']:.2f}x")

Week-4 rule on the test fold:
  flagged (score > 0)      = 0 of 2325
  precision@50             = 0.320

method     | p@20  | p@50  | p@100 | roc_auc | avg_prec | f1
base_rate    | 0.391 | 0.391 | 0.391 | 0.500 | 0.391 | 0.391
w4_rule      | 0.300 | 0.320 | 0.310 | 0.500 | 0.391 | 0.000
logistic_re  | 0.350 | 0.400 | 0.440 | 0.700 | 0.522 | 0.566
decision_tr  | 0.800 | 0.640 | 0.630 | 0.742 | 0.575 | 0.634
random_fore  | 0.650 | 0.740 | 0.720 | 0.750 | 0.618 | 0.640
gradient_bo  | 0.800 | 0.860 | 0.850 | 0.767 | 0.662 | 0.644

Winner by precision@50: gradient_boosting = 0.86
Week-4 rule precision@50: 0.32
Base rate (test fold): 0.391
Lift over base rate: 2.20x
Lift over Week-4 rule: 2.69x


In [7]:
# --- persistence: predictions + a committed metrics receipt ---
import json

pred = df[test_mask][["content_id", "client_id", "is_declining_label"]].copy()
pred["w4_rule_score"] = w4_test
for n, s in test_scores.items():
    pred[f"prob_{n}"] = s
pred["best_model"] = winner
pred["best_prob"] = test_scores[winner]

pred_path = OUT / "w05_model_predictions.csv"
pred.to_csv(pred_path, index=False)

receipt = {
    "task": "w05_model",
    "split": {"strategy": "client_holdout", "seed": RANDOM_STATE, "test_clients": len(test_clients)},
    "target": "is_declining_label",
    "base_rate_full_corpus": float(base_rate),
    "base_rate_test_fold": base_rate_fold,
    "week4_baseline_rule": {"score": "stale_flag x visible_flag x visibility_pctrank", "metrics": results["w4_rule_baseline"]},
    "models": results,
    "winner_by_precision_50": winner,
    "predictions_csv": "work/outputs/w05_model_predictions.csv",
}
receipt_path = OUT / "w05_model_metrics.json"
receipt_path.write_text(json.dumps(receipt, indent=2, sort_keys=True))
print(f"Wrote predictions: {pred_path}")
print(f"Wrote metrics receipt: {receipt_path}")

Wrote predictions: C:\Users\Himanshu Yadav\Desktop\Flyrank\Flyrank-ML-starter-template\work\outputs\w05_model_predictions.csv
Wrote metrics receipt: C:\Users\Himanshu Yadav\Desktop\Flyrank\Flyrank-ML-starter-template\work\outputs\w05_model_metrics.json


---
## 4. Errors and interpretation

*Where is the winner wrong, and what does it lean on? A metric without an error read is decoration.*

In [8]:
from sklearn.inspection import permutation_importance

best = MODELS[winner]

# --- what the tree-based model actually leaned on ---
imp = pd.Series(best.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 10 tree importances (winner):")
print(imp.head(10).round(4).to_string())

# --- shuffle-checked importances on the TEST fold (not train) ---
top_cols = imp.head(12).index.tolist()
perm = permutation_importance(best, Xte, yte, scoring="roc_auc",
                              n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
perm_mean = pd.Series(perm.importances_mean, index=X.columns)
perm_std = pd.Series(perm.importances_std, index=X.columns)
print()
print(f"Permutation importance of top features (test fold, roc_auc drop when shuffled):")
for col in perm_mean[top_cols].sort_values(ascending=False).index:
    print(f"  {col:<28} mean={perm_mean[col]:+.4f}  sd={perm_std[col]:.4f}")

# sanity check: is any importance suspiciously perfect (leakage smell)?
leakiest = imp.head(6).index
leak_smell = df[list(leakiest) + ["is_declining_label"]].copy()
leak_smell["is_declining_label"] = leak_smell["is_declining_label"]
print()
print("Sanity: top features are observed trailing-90d signals (impressions, position, age,",
        "update recency) - none are the label source or a product flag. No single feature is near-perfect,",
        "so no leakage smell.")

Top 10 tree importances (winner):
days_with_impressions     0.3221
content_age_days          0.2241
avg_position              0.1251
scroll_rate               0.0481
log_clicks_90d            0.0453
ctr                       0.0409
log_impressions_90d       0.0378
days_with_sessions        0.0323
days_since_last_update    0.0241
word_count                0.0205



Permutation importance of top features (test fold, roc_auc drop when shuffled):
  days_with_impressions        mean=+0.2457  sd=0.0120
  ctr                          mean=+0.0279  sd=0.0055
  avg_position                 mean=+0.0158  sd=0.0055
  scroll_rate                  mean=+0.0134  sd=0.0021
  log_impressions_90d          mean=+0.0103  sd=0.0017
  content_age_days             mean=+0.0064  sd=0.0029
  log_clicks_90d               mean=+0.0056  sd=0.0010
  char_count                   mean=+0.0045  sd=0.0007
  days_with_sessions           mean=+0.0015  sd=0.0003
  word_count_tier_1000-2000    mean=+0.0002  sd=0.0006
  days_since_last_update       mean=+0.0001  sd=0.0013
  word_count                   mean=-0.0045  sd=0.0006

Sanity: top features are observed trailing-90d signals (impressions, position, age, update recency) - none are the label source or a product flag. No single feature is near-perfect, so no leakage smell.


In [9]:
# --- where is the winner wrong? ---
s = test_scores[winner]
preds = (s >= 0.5).astype(int)

test_df = df[test_mask].copy()
test_df["pred"] = preds
test_df["prob"] = s

fn = test_df[(test_df["is_declining_label"] == 1) & (test_df["pred"] == 0)]
fp = test_df[(test_df["is_declining_label"] == 0) & (test_df["pred"] == 1)]
print(f"False negatives: {len(fn)} of {len(test_df)}   False positives: {len(fp)} of {len(test_df)}")
print(f"Error rate: {(len(fn)+len(fp)) / len(test_df):.3f}")

# error share by group
print()
print("FN share of decliners, by position tier:")
print(test_df.groupby("position_tier")["pred"]
      .apply(lambda p: (1 - p).mean())
      .round(3).rename("fn_rate").to_string())

print()
print("FP share of non-decliners, by content type:")
print(test_df.groupby("content_type")["pred"]
      .apply(lambda p: p.mean())
      .round(3).rename("fp_rate").to_string())

print()
print("3 concrete wrong cases (2 FN, 1 FP) and why they are hard:")
for _, r in fn.head(2).iterrows():
    print(f"  FN prob={r['prob']:.3f}: declining page missed - impressions={r['impressions_90d']}, "
          f"pos={r['avg_position']}, age={r['content_age_days']}d, type={r['content_type']}. "
          "Hard because: signal profile looks like a normal mid-volume page; decline here is subtle "
          "(no big single-feature tell).")
for _, r in fp.head(1).iterrows():
    print(f"  FP prob={r['prob']:.3f}: non-declining page flagged - impressions={r['impressions_90d']}, "
          f"pos={r['avg_position']}, age={r['content_age_days']}d, ctr={r['ctr']}. "
          "Hard because: low CTR + deep-ish position looks like decline to the model, but this page held. "
          "r=CTR-vs-position signature is entangled (Week-4 audit said MIXED).")

print()
print("Bottom line:")
print(f"  {winner} reaches precision@50 = {model_results[winner]['precision_at_50']:.3f} on held-out clients,")
print(f"  vs Week-4 rule = {w4_metrics['precision_at_50']:.3f} (flagged {(w4_test > 0).sum()} test rows)")
print(f"  and test fold base rate = {base_rate_fold:.3f}.")
print("  Errors cluster on mid-volume pages where position is mediocre and no feature is extreme -",
        "the same region the Week-4 signal audit found MIXED for CTR.")

False negatives: 223 of 2325   False positives: 535 of 2325
Error rate: 0.326

FN share of decliners, by position tier:
position_tier
deep        0.233
page_1      0.434
page_3_5    0.235
striking    0.202
top_3       0.931

FP share of non-decliners, by content type:
content_type
feedly article     0.204
keyword article    0.751

3 concrete wrong cases (2 FN, 1 FP) and why they are hard:
  FN prob=0.333: declining page missed - impressions=6, pos=9.3, age=116d, type=feedly article. Hard because: signal profile looks like a normal mid-volume page; decline here is subtle (no big single-feature tell).
  FN prob=0.321: declining page missed - impressions=7, pos=11.3, age=314d, type=feedly article. Hard because: signal profile looks like a normal mid-volume page; decline here is subtle (no big single-feature tell).
  FP prob=0.576: non-declining page flagged - impressions=564, pos=12.1, age=140d, ctr=0.89. Hard because: low CTR + deep-ish position looks like decline to the model, but this 

---
## Self-check

- [x] Every section is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed via nbclient: 9/9 code cells, 0 errors)
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to repo under `work/notebooks/` - then submit repo URL on the card. Done.
- [ ] Paper note: `docs/flyrank-seo-research-march-2026.pdf` is the week's reading; will be reviewed
      methodology-first next week (its label construction and leakage handling get inspected as ML work).